# Lecture 7. DocumentGPT

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.vectorstores import FAISS
from langchain.storage import LocalFileStore
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough, RunnableLambda
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

llm = ChatOpenAI(
    temperature=0.1,
)

ind_check_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            질문 텍스트에 개인정보가 있는지 확인해야합니다. 
            
            개인정보 형태는 아래와 같습니다.
            1. 주민등록번호 : 숫자 13자리로 구성되며, '-'문자가 중간에 있을 수 있습니다. 
            2. 이메일 : 영문 텍스트로 되어 있으며 '@'와 도메인이 포함되어 있습니다.
            
            개인정보가 있으면 
            답변 : ''
            -------
            """,
        ),
        ("human", "{question}"),
    ]
)

ind_check_chain = ind_check_prompt | llm
ind_check_chain.invoke({"question": RunnablePassthrough()})

final_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            질문 내용에 개인정보가 있는지 확인하고, 개인정보가 있다면 어떤 것이 위반인지 답변을 주어야 합니다.
            모르는 질문에는 답변하지 마세요.
            
            ------
            {context}
            """,
        ),
        ("human", "{question}"),
    ]
)

chain = {"context": ind_check_chain, "question": RunnablePassthrough()} | final_prompt | llm

chain.invoke("안녕하세요. 저는 1234561234567입니다.")

Retrying langchain.chat_models.openai.ChatOpenAI.completion_with_retry.<locals>._completion_with_retry in 4.0 seconds as it raised APIConnectionError: Error communicating with OpenAI: HTTPSConnectionPool(host='api.openai.com', port=443): Max retries exceeded with url: /v1/chat/completions (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1002)'))).


APIConnectionError: Error communicating with OpenAI: HTTPSConnectionPool(host='api.openai.com', port=443): Max retries exceeded with url: /v1/chat/completions (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1002)')))

In [3]:
import certifi
certifi.where()

'/Users/hwangms/Documents/workspace/LLM_Study/langchain_study/hwang/.venv/lib/python3.11/site-packages/certifi/cacert.pem'